# Data prep recipes — composable preprocessing

The `hmm_core.prep` module ships **21 atomic ops + 8 bundled YAML recipes** for common preprocessing tasks. Recipes are composable via `includes:`, fully reproducible via sidecar yaml, and produce HMM-ready (X, Z) splits.

**Engine is general-purpose** (pandas-thin), library covers both general (normalize, forward-fill) and HMM-canonical (log returns, vol features) workflows.

## Discover available recipes

In [ ]:
from hmm_core.prep import Pipeline, list_recipes

list_recipes()

## Use a bundled recipe by name

`crypto_basic_prep` is a composed recipe : log-returns + 20-day rolling vol + z-score + dropna. It produces an `output:` split into observations (X) and covariates (Z) directly usable by `fit_nhmm`.

In [ ]:
import numpy as np
import pandas as pd

# Synthetic OHLC-like data
rng = np.random.default_rng(0)
T = 300
raw = pd.DataFrame({
    "close": 100 * np.cumprod(1 + rng.normal(0.001, 0.015, T)),
    "volume": rng.uniform(1000, 5000, T),
}, index=pd.date_range("2024-01-01", periods=T, freq="D"))
raw.head()

In [ ]:
pipe = Pipeline.from_recipe("crypto_basic_prep")
pipe                                    # rich HTML view of the compiled pipeline

In [ ]:
prepared = pipe.fit_transform(raw)
prepared

In [ ]:
# X (observations) and Z (covariates) are ready for fit_nhmm
print("X shape:", prepared.X.shape if prepared.X is not None else None)
print("Z shape:", prepared.Z.shape if prepared.Z is not None else None)

## Build a custom pipeline in Python

The Python escape hatch is always available when YAML is too restrictive.

In [ ]:
pipe = (
    Pipeline()
    .add_step("log_diff", column="close", new_name="ret")
    .add_step("rolling_std", column="ret", window=10, new_name="vol_10")
    .add_step("rolling_std", column="ret", window=30, new_name="vol_30")
    .add_step("zscore", columns=["ret", "vol_10", "vol_30"])
    .add_step("dropna")
    .set_output(observations=["ret"], covariates=["vol_10", "vol_30"])
)
pipe

In [ ]:
prepared = pipe.fit_transform(raw)
prepared

## Save provenance to a sidecar

The `to_sidecar` method writes the applied recipe as a YAML file alongside the prepared dataset. Anyone reading the sidecar later can reproduce the transformation exactly.

In [ ]:
import tempfile
from pathlib import Path

with tempfile.TemporaryDirectory() as td:
    out_path = Path(td) / "prepared.parquet"
    out_path.touch()
    sidecar = prepared.to_sidecar(out_path)
    print(sidecar.read_text())

## Compose your own recipe YAML

You can `includes:` other recipes and chain them. See `src/hmm_core/prep/recipes/crypto_basic_prep.yaml` for an example.